# Task 2 tuned submission candidate

This local candidate starts from scored v9 (overall 0.74593) and replaces **only Task 2** with predictions from `t2_cat_tuned_weighted_full_v8`. The output is not a measured leaderboard improvement. Task 2's supplied `player_id` is separate from submission `ANS` IDs; as in the current assembly workflow, saved EDA Test row position is matched to sample submission row position after verifying raw Test order. Test is used only for final prediction and format checks.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import sys,json,hashlib,joblib,csv
import numpy as np
import pandas as pd
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
from src.pipeline.random_seed import set_seed
from src.task2.models.decision_rules import restore_categorical_nulls
SEED=set_seed()
BASE='kaggle_t12345_t5_alert25_v9'
NAME='kaggle_t12345_t2_tuned_v12'
MODEL='t2_cat_tuned_weighted_full_v8'
base_card=json.loads((SRC/'submission'/f'{BASE}.json').read_text(encoding='utf-8'))
base_path=SRC/base_card['submission_path']
assert hashlib.sha256(base_path.read_bytes()).hexdigest()==base_card['sha256']
base=pd.read_csv(base_path)
sample=pd.read_csv(SRC/'submission'/'sample_submission.csv')
expected=['id','task1','task2','task3','task4','task5']
assert base.columns.tolist()==sample.columns.tolist()==expected
assert len(base)==len(sample)==25889 and base.id.tolist()==sample.id.tolist()
assert sample.id.tolist()==[f'ANS{i:05d}' for i in range(1,25890)]
model_card=json.loads((SRC/'task2'/'save'/f'{MODEL}.json').read_text(encoding='utf-8'))
evaluation=json.loads((SRC/'task2'/'save'/f'{MODEL}-evaluation.json').read_text(encoding='utf-8'))
data_card=json.loads((SRC/model_card['data_card_path']).read_text(encoding='utf-8'))
assert model_card['name']==MODEL and model_card['data_name']==data_card['name']
assert evaluation['metric']=='macro_F1_cv5_procedure'
test=pd.read_parquet(SRC/data_card['splits']['test']['path'])
raw_ids=pd.read_csv(SRC/'task2'/'data'/'raw'/'test.csv',usecols=['player_id']).player_id
assert len(test)==len(base) and test.player_id.is_unique
assert test.player_id.reset_index(drop=True).equals(raw_ids.reset_index(drop=True))
assert base_card['sources']['task2']['test_id_first']==test.player_id.iloc[0]
assert base_card['sources']['task2']['test_id_last']==test.player_id.iloc[-1]
print('Verified base, model, data, and Test row order:',len(test))

Verified base, model, data, and Test row order: 25889


In [2]:
# Final Test prediction and controlled one-column substitution.
raw_X=test.drop(columns=['player_id'])
cats=raw_X.select_dtypes(include=['object']).columns.tolist()
X=restore_categorical_nulls(raw_X,cats)
model=joblib.load(SRC/model_card['model_path'])
pred=np.asarray(model.predict(X)).reshape(-1).astype(int)
assert len(pred)==len(base) and set(pred)<=set(range(4))
output=base.copy()
output['task2']=pred
assert output[['id','task1','task3','task4','task5']].equals(
    base[['id','task1','task3','task4','task5']])
assert output.id.tolist()==sample.id.tolist() and output.notna().all().all()
changed=int((output.task2!=base.task2).sum())
print('Changed Task 2 predictions:',changed,'/',len(output))
print('Prediction counts:',output.task2.value_counts().sort_index().to_dict())

Changed Task 2 predictions: 1088 / 25889
Prediction counts: {0: 10896, 1: 6160, 2: 5096, 3: 3737}


In [3]:
# Save the local candidate and its lineage once.
csv_path=SRC/'submission'/f'{NAME}.csv'
card_path=SRC/'submission'/f'{NAME}.json'
assert csv_path.exists()==card_path.exists(),'Inspect incomplete submission artifact'
with base_path.open('r',newline='',encoding='utf-8') as f:
    base_rows=list(csv.reader(f))
assert base_rows[0]==expected and len(base_rows)==len(output)+1
if csv_path.exists():
    old_card=json.loads(card_path.read_text(encoding='utf-8'))
    assert old_card['name']==NAME and old_card['base_sha256']==base_card['sha256']
    assert old_card['sha256']==hashlib.sha256(csv_path.read_bytes()).hexdigest()
    with csv_path.open('r',newline='',encoding='utf-8') as f:
        current_rows=list(csv.reader(f))
    # Finish this unscored draft if float serialization changed a non-Task-2 field.
    needs_fix=any([r[j] for j in (0,1,3,4,5)]!=[b[j] for j in (0,1,3,4,5)]
                  for r,b in zip(current_rows,base_rows))
else:
    needs_fix=True
if needs_fix:
    candidate_rows=[row.copy() for row in base_rows]
    for i,value in enumerate(pred,1):candidate_rows[i][2]=str(int(value))
    with csv_path.open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f,lineterminator='\n');writer.writerows(candidate_rows)
    sources=dict(base_card['sources'])
    sources['task2']={'model_name':MODEL,'model_card_path':f'task2/save/{MODEL}.json',
        'model_path':model_card['model_path'],'data_name':data_card['name'],
        'data_card_path':model_card['data_card_path'],
        'test_path':data_card['splits']['test']['path'],
        'evaluation_metric':evaluation['metric'],'evaluation_score':evaluation['score'],
        'evaluation_note':'Five-fold development procedure estimate; full fit not directly scored',
        'test_rows':len(test),'test_id_first':test.player_id.iloc[0],
        'test_id_last':test.player_id.iloc[-1]}
    detail={'name':NAME,'created_at':datetime.now(timezone.utc).isoformat(),
        'card':'Full-Train tuned weighted Task 2 candidate; Tasks 1, 3, 4, 5 identical to scored v9',
        'submission_path':f'submission/{NAME}.csv','format':'comma-separated CSV',
        'rows':len(output),'columns':expected,'sha256':hashlib.sha256(csv_path.read_bytes()).hexdigest(),
        'base_submission':BASE,'base_sha256':base_card['sha256'],'changed_tasks':[2],
        'task2_changed_rows':changed,
        'task2_id_mapping':'Saved EDA Test row position to sample ANS ID; raw Test order verified',
        'tier_timing':'unverified','sources':sources}
    card_path.write_text(json.dumps(detail,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
else:
    detail=json.loads(card_path.read_text(encoding='utf-8'))
    assert detail['base_sha256']==base_card['sha256'] and detail['task2_changed_rows']==changed
    assert detail['sha256']==hashlib.sha256(csv_path.read_bytes()).hexdigest()
saved=pd.read_csv(csv_path)
with csv_path.open('r',newline='',encoding='utf-8') as f:
    final_rows=list(csv.reader(f))
assert all([r[j] for j in (0,1,3,4,5)]==[b[j] for j in (0,1,3,4,5)]
           for r,b in zip(final_rows,base_rows))
assert saved.columns.tolist()==expected and saved.id.tolist()==output.id.tolist()
for col in ('task1','task2','task4','task5'):
    assert np.array_equal(saved[col].to_numpy(),output[col].to_numpy())
assert np.allclose(saved.task3.to_numpy(),output.task3.to_numpy(),rtol=0,atol=1e-8)
print('Saved',csv_path,'SHA-256',detail['sha256'])

Saved

 C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_t2_tuned_v12.csv SHA-256 13329cce19f043b225ed357c3fd7e3a7839f9e5f59d2df5492eb72dac3acd0a2
